In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor



In [ ]:
df = pd.read_csv('/content/data.csv')

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (4600, 18)
                  date      price  bedrooms  bathrooms  sqft_living  sqft_lot  \
0  2014-05-02 00:00:00   313000.0       3.0       1.50         1340      7912   
1  2014-05-02 00:00:00  2384000.0       5.0       2.50         3650      9050   
2  2014-05-02 00:00:00   342000.0       3.0       2.00         1930     11947   
3  2014-05-02 00:00:00   420000.0       3.0       2.25         2000      8030   
4  2014-05-02 00:00:00   550000.0       4.0       2.50         1940     10500   

   floors  waterfront  view  condition  sqft_above  sqft_basement  yr_built  \
0     1.5           0     0          3        1340              0      1955   
1     2.0           0     4          5        3370            280      1921   
2     1.0           0     0          4        1930              0      1966   
3     1.0           0     0          4        1000           1000      1963   
4     1.0           0     0          4        1140            800      1976   

   yr_renova

In [ ]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract useful features from date
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Drop original date column
df.drop('date', axis=1, inplace=True)

# Drop columns that are not useful for prediction
df.drop(['street', 'country'], axis=1, inplace=True)

# Handle categorical columns
label_cols = ['city', 'statezip']

le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])

# Check missing values
print(df.isnull().sum())

# Fill missing values if any
df.fillna(df.median(), inplace=True)

price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
city             0
statezip         0
year             0
month            0
dtype: int64


In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training Samples:", X_train.shape)
print("Testing Samples:", X_test.shape)

Training Samples: (3680, 16)
Testing Samples: (920, 16)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "Gradient Boosting": GradientBoostingRegressor()
}

In [ ]:
results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = [rmse, r2]

    print("\nModel:", name)
    print("RMSE:", rmse)
    print("R2 Score:", r2)


Model: Linear Regression
RMSE: 992723.2404236451
R2 Score: 0.03367851279125411

Model: Decision Tree
RMSE: 1005974.3262509392
R2 Score: 0.007708999583460052

Model: Random Forest
RMSE: 987733.1795149483
R2 Score: 0.043368794330795546

Model: Gradient Boosting
RMSE: 983572.3814420921
R2 Score: 0.051411382764599445


In [ ]:
results_df = pd.DataFrame(results, index=["RMSE","R2 Score"]).T
print("\nModel Comparison")
print(results_df)


Model Comparison
                           RMSE  R2 Score
Linear Regression  9.927232e+05  0.033679
Decision Tree      1.005974e+06  0.007709
Random Forest      9.877332e+05  0.043369
Gradient Boosting  9.835724e+05  0.051411


In [ ]:
from sklearn.ensemble import VotingRegressor, StackingRegressor

In [ ]:
df = pd.read_csv('/content/data.csv')

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

df.drop('date', axis=1, inplace=True)
df.drop(['street', 'country'], axis=1, inplace=True)

label_cols = ['city', 'statezip']
le = LabelEncoder()

for col in label_cols:
    df[col] = le.fit_transform(df[col])

df.fillna(df.median(), inplace=True)

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
# Base Models
lr = LinearRegression()
dt = DecisionTreeRegressor()
rf = RandomForestRegressor(n_estimators=100)
gb = GradientBoostingRegressor()

In [ ]:
# Train base models
models = [lr, dt, rf, gb]

for model in models:
    model.fit(X_train, y_train)

In [ ]:
# 1 Voting Regressor Ensemble

voting_model = VotingRegressor(
    estimators=[
        ('lr', lr),
        ('dt', dt),
        ('rf', rf),
        ('gb', gb)
    ]
)

voting_model.fit(X_train, y_train)

y_pred_voting = voting_model.predict(X_test)

rmse_voting = np.sqrt(mean_squared_error(y_test, y_pred_voting))
r2_voting = r2_score(y_test, y_pred_voting)

print("Voting Regressor Results")
print("RMSE:", rmse_voting)
print("R2 Score:", r2_voting)

Voting Regressor Results
RMSE: 987322.8280422372
R2 Score: 0.044163489678056056


In [ ]:
# 2 Stacking Regressor Ensemble

stacking_model = StackingRegressor(
    estimators=[
        ('dt', dt),
        ('rf', rf),
        ('gb', gb)
    ],
    final_estimator=LinearRegression()
)

stacking_model.fit(X_train, y_train)

y_pred_stack = stacking_model.predict(X_test)

rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
r2_stack = r2_score(y_test, y_pred_stack)

print("\nStacking Regressor Results")
print("RMSE:", rmse_stack)
print("R2 Score:", r2_stack)


Stacking Regressor Results
RMSE: 985532.7447687965
R2 Score: 0.04762634052429471
